# Reranker 점수 계산 (Colab GPU)

로컬 CPU로는 Qwen3-Reranker 채점이 1시간 넘게 걸려서, (쿼리, 가맹점) 쌍의 reranker 점수만 GPU에서 계산합니다. 비교·채점은 로컬에서 합니다.

**서비스셋(1-3 reranker, `docs/SERVICE_PIPELINE_PLAN.md`)**: 로컬 `python scripts/eval_service_pipeline.py --exp rerank --export-pairs`
→ `results/service_pipeline/rerank/rerank_pairs.csv`를 Drive `내 드라이브/store-search-ai/stack_eval/service/`에 올리고 `SPLIT = "service"`로 실행
→ 생긴 `rerank_scores_<모델>.csv`를 로컬 `results/service_pipeline/rerank/`에 넣고 `--exp rerank`.

**기존 벤치마크(stack_eval)**:

1. 로컬: `python scripts/eval_retrieval_stack.py --dense qwen3_embedding_0_6b qwen3_embedding_0_6b_ft_kse14_20261014_0440 --export-pairs`
   → `results/stack_eval/<split>/rerank_pairs.csv`를 Drive `내 드라이브/store-search-ai/stack_eval/<split>/`에 올림(test면 `--split test --dense-dir ...`)
2. Colab: 런타임 유형 **GPU(T4)** → `SPLIT` 확인 → 위에서부터 실행 → 같은 Drive 폴더에 `rerank_scores_<모델>.csv`가 생김(이미 있으면 건너뜀)
3. 로컬: 그 파일을 `results/stack_eval/<split>/`에 넣고 1번 명령을 `--export-pairs` 없이 `--rerankers ...`로 실행 → 캐시를 읽어 비교표 출력

채점 함수(`load_reranker`)는 `scripts/eval_retrieval_stack.py`와 같은 코드입니다(테스트로 확인). 고치려면 스크립트를 고친 뒤 이 셀에 그대로 복사하세요.

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SPLIT = "service"   # "service"(서비스셋) / "val"·"test"(기존 벤치마크) — 로컬에서 --export-pairs로 만든 쌍이 있는 폴더
STACK_DIR = Path("/content/drive/MyDrive/store-search-ai/stack_eval") / SPLIT
RERANKERS = ["BAAI/bge-reranker-v2-m3", "Qwen/Qwen3-Reranker-0.6B"]   # 서비스셋은 둘 다(T4에서 4만 쌍 기준 합계 10~20분 예상)

In [ ]:
# @cell reranker
def load_reranker(name: str):
    """(query, doc) 쌍 목록 → 점수 배열 함수. bge 계열은 CrossEncoder, Qwen3-Reranker는 공식 yes/no 방식."""

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if "qwen3-reranker" not in name.lower():
        from sentence_transformers import CrossEncoder

        model = CrossEncoder(name, max_length=256, device=device)
        return lambda pairs: model.predict(pairs, batch_size=64, show_progress_bar=True)

    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device).eval()
    yes, no = tok.convert_tokens_to_ids("yes"), tok.convert_tokens_to_ids("no")
    prefix = ("<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct "
              "provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n")
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    instruct = "Given a Korean local-store search query, judge whether the store satisfies the user's shopping, dining, or service intent"

    def score(pairs):
        # 마지막 토큰의 yes/no 두 개 logit만 필요하다 — 출력층(어휘 15만 개)을 전체 토큰에 돌리면 계산이 수백 배가 되므로
        # 마지막 hidden state에 yes/no 두 행만 곱한다(왼쪽 padding이라 마지막 위치가 곧 각 문장의 끝).
        head = model.lm_head.weight[[no, yes]]
        out = []
        for i in range(0, len(pairs), 32):
            texts = [f"{prefix}<Instruct>: {instruct}\n<Query>: {q}\n<Document>: {d}{suffix}" for q, d in pairs[i:i + 32]]
            batch = tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                hidden = model.model(**batch).last_hidden_state[:, -1]
                logits = hidden @ head.T
            out.extend(torch.log_softmax(logits, dim=-1)[:, 1].exp().cpu().tolist())
            if (i // 32) % 20 == 0:
                print(f"  {min(i + 32, len(pairs))}/{len(pairs)}", flush=True)
        return np.asarray(out)

    return score

In [ ]:
import gc

pairs = pd.read_csv(STACK_DIR / "rerank_pairs.csv", encoding="utf-8-sig")
print(f"채점할 쌍 {len(pairs)}개")
for name in RERANKERS:
    out = STACK_DIR / f"rerank_scores_{name.split('/')[-1]}.csv"
    if out.exists():
        old = pd.read_csv(out, encoding="utf-8-sig")
        if set(zip(pairs["query_id"], pairs["doc_id"])) <= set(zip(old["query_id"], old["doc_id"])):   # 행 수가 아니라 쌍이 다 있는지
            print(f"{out.name}: 이미 모든 쌍이 있음 — 건너뜀")
            continue
    scores = load_reranker(name)(list(zip(pairs["query"], pairs["doc_text"])))
    pairs.assign(score=np.asarray(scores, dtype=float))[["query_id", "doc_id", "score"]].to_csv(
        out, index=False, encoding="utf-8-sig", lineterminator="\n")
    print(f"[저장] {out}")
    gc.collect()
    torch.cuda.empty_cache()